<a href="https://colab.research.google.com/github/suflorian-commits/modeling_project/blob/main/preprocessing/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the final project for environmental modeling course. This project will try to predict binary JLM rain events based on data from beit dagan station.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:

!pwd
!ls
!ls /content/drive
!ls /content/drive/MyDrive/Modeling_Project/Data

/content
drive  sample_data
MyDrive  Othercomputers  Shareddrives
'Beit Dagan'   Jerusalem


In [ ]:
# necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

The data is obtained from Israel's Met Office in 10-min resoultion. For jeruslam: time and rain amonut (mm). For Beit Dagan: Pressure (hPa), RH[%],Temp[C],Wind direction (deg), Wind Velocisty [m/sec], std for wind direction (deg), rain amount (mm).

In [ ]:
# reading the data for the project
filepath = '/content/drive/MyDrive/Modeling_Project/Data/'
BD_file = 'Beit Dagan/BD_P_RH_T_windDir_windSpeed_windDirSTD_rain_20101001_20250930.csv'
JLM_file = 'Jerusalem/jer_rain_20101001_20250930.csv'
BD_original_data = pd.read_csv(filepath + BD_file)
JLM_original_data = pd.read_csv(filepath+JLM_file)

/tmp/ipython-input-39540855.py:5: DtypeWarning: Columns (2,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  BD_original_data = pd.read_csv(filepath + BD_file)


In [ ]:
BD_data = BD_original_data.copy()

# Rename columns
BD_data.columns = [
    'station', 'time', 'pressure', 'RH', 'temp',
    'wind_dir', 'wind_speed', 'wind_dir_std', 'rain_amount'
]

# Strip whitespace from 'time'
BD_data['time'] = BD_data['time'].str.strip()

# Convert 'time' to datetime
BD_data['time'] = pd.to_datetime(
    BD_data['time'],
    format='%d/%m/%Y %H:%M',
    errors='coerce'
)

# Convert numeric columns
numeric_cols = ['pressure', 'RH', 'temp', 'wind_dir', 'wind_speed', 'wind_dir_std', 'rain_amount']
for col in numeric_cols:
    BD_data[col] = pd.to_numeric(
        BD_data[col].astype(str).str.replace(',', '.'),
        errors='coerce'
    )

# Filter to winter months
BD_data = BD_data[BD_data['time'].dt.month.isin([10,11,12,1,2,3])]

# Group by hour
BD_hourly = BD_data.groupby(BD_data['time'].dt.floor('h')).agg({
    'rain_amount': 'sum'
}).reset_index()

# Binary rain column
BD_hourly['rain_binary'] = (BD_hourly['rain_amount'] > 0).astype(int)

In [ ]:
JLM_data = JLM_original_data.copy()
# Rename columns
JLM_data.columns = ['station', 'time', 'rain_amount']

# Strip whitespace from 'time'
JLM_data['time'] = JLM_data['time'].str.strip()

# Convert 'time' to datetime
JLM_data['time'] = pd.to_datetime(
    JLM_data['time'],
    format='%d/%m/%Y %H:%M',
    errors='coerce'
)

# Convert numeric columns
JLM_data['rain_amount'] = pd.to_numeric(
    JLM_data['rain_amount'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

# Filter to winter months
JLM_data = JLM_data[JLM_data['time'].dt.month.isin([10,11,12,1,2,3])]

# Group by hour (WITHOUT creating fictitious hours)
JLM_hourly = JLM_data.groupby(JLM_data['time'].dt.floor('h')).agg({
    'rain_amount': 'sum'
}).reset_index()

# Binary rain column
JLM_hourly['rain_binary'] = (JLM_hourly['rain_amount'] > 0).astype(int)

In [ ]:
print(len(JLM_original_data))
print(len(BD_original_data))

786293
787977


In [ ]:
def align_and_clean_dataframes(jlm_hourly, bd_hourly, dt, print_summary=True):
    """
    Vectorized version - aligns JLM and BD dataframes by removing rows with missing data.

    Parameters:
    -----------
    jlm_hourly : DataFrame
        Jerusalem hourly data with 'time' column
    bd_hourly : DataFrame
        BD hourly data with 'time' column
    dt : int
        Time lag in hours (JLM at time t, BD at time t+dt)
    print_summary : bool
        Switch to print summary of length of original data and cleaned data, default = True
    Returns:
    --------
    tuple : (jlm_cleaned, bd_cleaned)
    """
    # Ensure time is datetime
    jlm = jlm_hourly.copy()
    bd = bd_hourly.copy()
    jlm['time'] = pd.to_datetime(jlm['time'])
    bd['time'] = pd.to_datetime(bd['time'])

    # Create shifted BD times (subtract dt to align with JLM times)
    bd_shifted = bd.copy()
    bd_shifted['time_shifted'] = bd_shifted['time'] - pd.Timedelta(hours=dt)

    # Find BD rows with complete data (no NaN in any column)
    bd_complete_mask = ~bd_shifted.drop(columns=['time', 'time_shifted']).isna().any(axis=1)
    bd_complete = bd_shifted[bd_complete_mask]

    # Merge JLM with complete BD data on aligned times
    merged = jlm.merge(
        bd_complete[['time_shifted']],
        left_on='time',
        right_on='time_shifted',
        how='inner'
    )

    # Get valid times
    valid_jlm_times = merged['time'].values
    valid_bd_times = (merged['time'] + pd.Timedelta(hours=dt)).values

    # Filter original dataframes
    jlm_clean = jlm[jlm['time'].isin(valid_jlm_times)].reset_index(drop=True)
    bd_clean = bd[bd['time'].isin(valid_bd_times)].reset_index(drop=True)

    # Print summary
    if print_summary:
        print(f"Original JLM rows: {len(jlm_hourly)}, Cleaned: {len(jlm_clean)}")
        print(f"Original BD rows: {len(bd_hourly)}, Cleaned: {len(bd_clean)}")

    return jlm_clean, bd_clean

In [ ]:
JLM_cleaned, BD_cleaned = align_and_clean_dataframes(JLM_hourly, BD_hourly, dt=6)

Original JLM rows: 65616, Cleaned: 61089
Original BD rows: 61173, Cleaned: 61089
